# Linear regression

*Fitting, reading, and evaluating a model of customer revenue*

In [Chapter 7](https://pyba.murtaza.cc/parts/part-03-statistical/ch-07-inference.html) we answered a yes-or-no question: did the pilot work? Most business questions have more structure. Which variables affect customer revenue, and by how much, once the other variables are taken into account? **Linear regression** is the standard tool for such questions in business functions such as marketing, finance, and operations. In this chapter we model Prairie Wholesale's twelve-month customer revenue, learn how to read a regression table the way an analyst does, and quantify the effect of one extreme customer on the fitted line.

> **Setup for this chapter**
>
> You can run this chapter's notebook in two ways. In the cloud, [**open ch-08-regression.ipynb in Google Colab**](https://colab.research.google.com/github/murtaza-nasir/pyba-companion/blob/main/notebooks/ch-08-regression.ipynb); nothing needs to be installed. Locally, use the `pyba-core` environment (Appendix A). Either way, run the setup cell below first. No keys or paid accounts are needed anywhere in this book; Colab requires only a free Google account.

In [ ]:
# Setup. Run this cell once per session. It installs this chapter's
# packages; on Google Colab it also fetches the course data.
%pip install -q pandas plotly statsmodels
import sys
if "google.colab" in sys.modules:
    !test -d pyba-companion || git clone --quiet --depth 1 https://github.com/murtaza-nasir/pyba-companion.git
    sys.path.insert(0, "pyba-companion")   # makes `import pyba` (DATA_DIR) work

> **Tools in this chapter**
>
> | Tool | Why we use it here | Alternatives | Trade-off |
> |---|---|---|---|
> | statsmodels | Regression with full statistical output, including coefficients, standard errors, and p-values | scikit-learn (prediction-focused, minimal statistics), R | statsmodels is built around statistical output, the focus of this chapter; scikit-learn is built around prediction |
>
> : {tbl-colwidths="[12,33,22,33]"}

## Regression with one predictor {#sec-ch8-intro}

Let us start by modeling the customer snapshot's `revenue_12m` variable, each customer's revenue over the past twelve months. If we were to predict a customer's revenue from the variables we have, the most obvious first predictor would be order frequency. To fit the model we use **statsmodels**, the statistics library of the Python ecosystem, an open-source project with origins in econometrics. Its formula interface accepts a model written the way statisticians write them: the outcome on the left, a tilde, and the predictors on the right, as in `revenue_12m ~ orders_12m`.

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import statsmodels.formula.api as smf

from pyba import DATA_DIR

snap = pd.read_csv(DATA_DIR / "pw_customer_snapshot.csv")

model_1 = smf.ols("revenue_12m ~ orders_12m", data=snap).fit()
print(f"intercept {model_1.params['Intercept']:.1f}, "
      f"slope {model_1.params['orders_12m']:.1f}, R² {model_1.rsquared:.3f}")

In [ ]:
px.scatter(snap, x="orders_12m", y="revenue_12m", opacity=0.45, trendline="ols",
           trendline_color_override="#0969da",
           labels={"orders_12m": "Orders (12 months)",
                   "revenue_12m": "Revenue (12 months, $)"})

The fitted line is chosen to minimize the sum of the squared vertical distances to the points. These vertical distances are the **residuals**: each customer's actual revenue minus the revenue the line predicts for their order count. The fitted line is summarized by three numbers. The **intercept** is the predicted revenue at zero orders; in this case it is an extrapolation with no business meaning. The **slope** is the estimate of interest: an additional order per year is associated with about \$450 more in revenue. **R²** is the share of the variance in revenue that the model accounts for, here 0.29. The name and the notation come from Pearson's r: with a single predictor, R² is the square of the correlation between the two variables. In [Chapter 5](https://pyba.murtaza.cc/parts/part-02-descriptive/ch-05-relationships.html) we measured this correlation at r = 0.54, and squaring 0.54 gives us the 0.29 figure we see here.

::: {.content-visible when-format="html"}
The explorer below shows the mechanics. Try dragging any point and let the line refit. The amber segments are the residuals: each one is the vertical difference between a data point and the value the regression line predicts at that point's position on the x-axis. Click "Add an outlier" and drag it to the corners of the plot; you will see the slope readout change.

<iframe src="../../assets/demos/least-squares.html" width="100%" height="450" style="border:1px solid #d0d7de; border-radius:8px;" title="Least squares explorer"></iframe>
:::

::: {.content-visible unless-format="html"}
The online edition has an interactive explorer here: twelve draggable points with a live least-squares line, visible residuals, and an optional outlier point that shows how far one observation can move the slope.
:::

## Reading the regression table

Calling `summary()` on the fitted model prints the full inference table, the standard statistical report of a regression. The table is dense with numbers, but only four of its columns are needed to read the model. We go through those four columns below the table.

In [ ]:
model_1.summary().tables[1]

- **coef** is the estimate: \$449 of revenue for every additional order.
- **std err** is the coefficient's standard error, its sampling variation as described in [Chapter 7](https://pyba.murtaza.cc/parts/part-03-statistical/ch-07-inference.html). Another sample of customers would yield a slightly different slope. The standard error specifies how different.
- **P\>\|t\|** is the p-value for the null hypothesis that the true coefficient is zero. [Chapter 7](https://pyba.murtaza.cc/parts/part-03-statistical/ch-07-inference.html)'s four reading rules apply here.
- **\[0.025, 0.975\]** is the 95% confidence interval. It runs from about \$403 to \$495, and each endpoint sits about \$46 from the \$449 estimate (roughly two standard errors), so the slope is estimated to within plus or minus \$46.

Everything in this table is inference about association. The slope describes how revenue differs between customers who differ by an order. Whether an extra order from a customer (resulting from a promotion, perhaps) would yield \$449 is a causal question we cannot answer with this regression. Causal methods for observational data exist, but they require strong additional assumptions and are beyond the scope of this book. The most direct route to a causal answer is a randomized design like the [Chapter 7](https://pyba.murtaza.cc/parts/part-03-statistical/ch-07-inference.html) pilot.

## Regression with multiple predictors

Order frequency alone explains 29% of the variance in revenue. Customers also differ in order size, tenure, and business type. Regression with more than one predictor, called **multiple regression**, estimates each variable's association with the outcome while holding the other predictors constant:

In [ ]:
model_2 = smf.ols(
    "revenue_12m ~ orders_12m + avg_order_value + tenure_months + C(business_type)",
    data=snap,
).fit()
print(f"R² {model_2.rsquared:.3f}")
model_2.summary().tables[1]

R² increases from 0.29 to 0.81. To interpret the coefficients, we now need one discipline above all: **each coefficient is the association with revenue when the other variables in the model are held constant**. The `orders_12m` coefficient (\~\$500) is the revenue difference between customers with the same average order value, tenure, and business type who differ by one order. This "holding fixed" reading extends an idea from [Chapter 5](https://pyba.murtaza.cc/parts/part-02-descriptive/ch-05-relationships.html). There, the pooled correlation between discounts and order totals was misleading, and we removed the channel's influence by recomputing the correlation within each channel separately. Multiple regression reaches the same goal without the splitting: instead of literally dividing the customers into groups, the model adjusts each comparison for the other variables. Statisticians call this **adjustment**.

The `C(business_type)` rows are read differently. A categorical predictor is introduced as **dummy variables**, one for each category, with one category omitted as the **reference**. statsmodels uses `clinic` as the reference, the alphabetically first category. Each business-type coefficient is that type's average difference from clinics, holding the numeric variables constant. The printed coefficients depend on which category is the reference: with `office` as the reference instead of `clinic`, every business-type coefficient would be measured against offices and the numbers would change. The fitted model itself would not change: it would produce the same predictions and the same gaps between the types. The choice of reference therefore affects only how the results are presented.

::: {.content-visible when-format="html"}
The explorer below refits the chapter's model under every possible reference. If you change the reference, you will see the coefficients shift while the gaps between types stay fixed.

<iframe src="../../assets/demos/reference-category.html" width="100%" height="440" style="border:1px solid #d0d7de; border-radius:8px;" title="reference category"></iframe>
:::

::: {.content-visible unless-format="html"}
The online edition has an explorer here: a dropdown sets the reference category and the business-type coefficients are recomputed around it, with the between-type gaps fixed.
:::


> **Python note: formula strings**
>
> The formula `"revenue_12m ~ orders_12m + C(business_type)"` is a mini-language in itself (borrowed from R). `+` adds predictors, `C()` denotes a column as categorical, and the intercept is included automatically. With formulas, model definitions stay readable. You will use them again in the labs.

## The influential customer

Under [Chapter 4](https://pyba.murtaza.cc/parts/part-02-descriptive/ch-04-single-variable.html)'s outlier policy, an extreme value that is genuine rather than an error stays in the data, and we disclose whenever a single observation moves any reported numbers. Regression requires the same discipline, because a single point far from the rest can pull the fitted line toward it. The least-squares explorer earlier in the chapter showed this: dragging a point at either corner of the plot in the direction opposite to the observed trend moved the slope far more than dragging any central point. Points with extreme values of the predictor have **leverage**; a high-leverage point whose outcome is also unusual is **influential**.

The snapshot has a candidate: the school district from [Chapter 4](https://pyba.murtaza.cc/parts/part-02-descriptive/ch-04-single-variable.html), which appears here as the customer with \$94,000 of annual revenue from only 12 orders. We measure its influence by fitting the model with and without it:

In [ ]:
top = snap["revenue_12m"].idxmax()
snap.loc[[top], ["customer_id", "business_type", "revenue_12m",
                 "orders_12m", "avg_order_value"]]

In [ ]:
without = snap.drop(index=top)

m_all = smf.ols("revenue_12m ~ avg_order_value", data=snap).fit()
m_without = smf.ols("revenue_12m ~ avg_order_value", data=without).fit()

pd.DataFrame({
    "slope ($ per $1 of AOV)": [m_all.params["avg_order_value"],
                                m_without.params["avg_order_value"]],
    "R²": [m_all.rsquared, m_without.rsquared],
}, index=["all 872 customers", "without the school district"]).round(3)

With this single customer included, a row that is 0.1% of the data, the slope is about 6% higher. We used a one-predictor model here so that the whole effect is absorbed by a single coefficient; in the multiple regression the effect would be split across coefficients, and each would move less. The residual table below shows what remains true in the full model: the model underpredicts this customer's revenue by about \$46,000, the largest residual in the data by a factor of two. Our policy for handling this influential customer is the same as [Chapter 4](https://pyba.murtaza.cc/parts/part-02-descriptive/ch-04-single-variable.html)'s outlier policy: the customer is a genuine account, so we keep it in the model. Any coefficient from the model presented in a report is accompanied by the with-and-without check.

In [ ]:
snap.assign(residual=model_2.resid).nlargest(3, "residual")[
    ["customer_id", "business_type", "revenue_12m", "residual"]].round(0)

## Prediction and its limits

We have been using the model's predictions throughout this chapter: every residual was an actual value minus a predicted one. So far, though, every prediction was for a customer already in the data. We can also apply the model to a new unseen customer. Here is the expected revenue for a hypothetical new account, a restaurant whose predictor values are typical of the restaurants in the data:

In [ ]:
new_customer = pd.DataFrame({
    "orders_12m": [20], "avg_order_value": [400],
    "tenure_months": [24], "business_type": ["restaurant"],
})
print(f"predicted 12-month revenue: ${model_2.predict(new_customer)[0]:,.0f}")

This prediction has two limitations. First, it is a conditional average. Individual customers with these characteristics will differ from it substantially: R² = 0.81 still corresponds to \$4,000 of average error (calculated below). Second, predictions are only reliable within the range of the data. The data contain no customer with 200 orders per year, so a prediction at that range extrapolates beyond anything observed. In this chapter, prediction has been a byproduct: our goal was inference, meaning the estimation and careful reading of the associations in the fitted model. In [Chapter 9](https://pyba.murtaza.cc/parts/part-04-predictive/ch-09-classification-1.html) we make prediction the objective itself, with a procedure to measure prediction quality on held-out data.

## Evaluation: better than what, by how much?

The claim under evaluation is that the model can forecast a customer's revenue. A forecast is useful only if it improves on what the business would do without it, so we compare the model against the simplest alternative: forecasting every customer at the overall mean revenue.

|  |  |
|---|---|
| **Metric** | root mean squared error (RMSE), the typical size of a prediction error, in dollars. |
| **Test** | the fitted data (in [Chapter 9](https://pyba.murtaza.cc/parts/part-04-predictive/ch-09-classification-1.html) we add the holdout version of this measurement). |
| **Baseline** | the no-model prediction, every customer forecast at the overall mean. |

: {tbl-colwidths="[18,82]"}

**Root mean squared error (RMSE)** condenses all the prediction errors into one number, computed in three steps: square each residual, average the squares, and take the square root of that average. The result is in the outcome's own units, dollars here, and because the errors are squared before averaging, large errors count disproportionately. The baseline needs no separate computation: when every customer is forecast at the mean, each error is a deviation from the mean, so the baseline's RMSE equals the standard deviation of revenue. The `ddof=0` argument selects the population version of the standard deviation, which is appropriate for this identity.

::: {.content-visible when-format="html"}
In the explorer below, we present a hypothetical variable with fixed actual values and editable predicted values. Compare the two error presets: they have the same average absolute error, yet very different RMSEs. Then enter one very large value in any predicted cell; you will see the RMSE respond far more than the average absolute error.

<iframe src="../../assets/demos/rmse-explorer.html" width="100%" height="420" style="border:1px solid #d0d7de; border-radius:8px;" title="RMSE explorer"></iframe>
:::

::: {.content-visible unless-format="html"}
The online edition has an RMSE explorer here: eight fixed actual values with editable predictions and live RMSE and average-absolute-error readouts. Its presets show that eight $100 errors and one $800 error have the same average absolute error but RMSEs of $100 and $283.
:::

In [ ]:
rmse_model = np.sqrt((model_2.resid ** 2).mean())
rmse_baseline = snap["revenue_12m"].std(ddof=0)

pd.Series({
    "baseline RMSE (predict the mean)": rmse_baseline,
    "model RMSE": rmse_model,
    "error reduction": f"{1 - rmse_model / rmse_baseline:.0%}",
})

The baseline's predictions are off by about \$9,200 per customer on average, and the model's by about \$4,000, a 56% reduction. This is the way we report every model in this course: against the baseline the business would otherwise use.

## The decision this informs

The account team currently ranks its customers by revenue alone. The revenue number is a blend of order frequency and order size, and newer accounts rank low on it simply because they are new. The regression model separates the drivers: order frequency and order size dominate, tenure adds about \$6 per month, and a business-type difference remains after adjustment. Schools are about \$2,200 below clinics at the same order count and order value, because school revenue comes through order size, and the model already accounts for order size. The concrete change is in the account reviews. The team now flags customers whose actual revenue is well below the model's prediction for their profile, because those accounts are underperforming given their own characteristics. Customers with the largest positive residuals receive a different flag, for concentration risk: their revenue is far above what their profile predicts, and revenue that ordinary characteristics do not account for may not recur. The school district from earlier in the chapter is the largest such case.

## Exercises



### Build lab

So far the outcome has been total revenue. Now model *average order value*: `avg_order_value ~ pct_online + avg_discount_pct + tenure_months + C(business_type)`. Report the fitted table, interpret the `pct_online` and `avg_discount_pct` coefficients in one sentence each (with the "holding fixed" discipline), and state which business types differ most from the reference after adjustment.

### Evaluate lab

Evaluate your build-lab model with the procedure we used in this chapter: compute its RMSE and the predict-the-mean baseline RMSE, and report the error reduction. Then drop the customer with the largest residual, refit the model, and report which coefficient moved most as a percentage. Answer in two sentences: is the model useful, and is it stable?

> **Lab starter**
>
> A starter notebook for this lab is provided. It restates the task, reproduces the objects from this chapter that the lab builds on, and marks the cells you complete. [**Open ch-08-lab-starter.ipynb in Google Colab**](https://colab.research.google.com/github/murtaza-nasir/pyba-companion/blob/main/notebooks/ch-08-lab-starter.ipynb), or download it from the course page in Blackboard. Colab opens a notebook from GitHub read-only: click **Copy to Drive** in the toolbar before you edit anything, and work in that copy.